# Formula-SWCT Generalization on Arabic Handwritten Digits

This notebook evaluates the final Formula-SWCT method on the Arabic Handwritten Digits dataset using an ImageNet-pretrained ResNet-18.

Arabic Digits introduces a substantial domain shift from the natural-image datasets used during Formula-SWCT calibration.

The evaluation compares:

- Baseline frozen ResNet-18,
- original Single-Parameter Curvature Tuning (S-CT),
- Formula-SWCT.

Formula-SWCT follows the final predict-first, train-once protocol:

1. Fit the Formula-SWCT predictor using only the calibration datasets.
2. Collect stage statistics from the Arabic Digits training split.
3. Predict four stage-specific beta values before downstream training.
4. Freeze the predicted beta values.
5. Extract fixed train, validation, and test features once.
6. Train one downstream linear classifier.
7. Select the classifier checkpoint using validation accuracy.
8. Evaluate once on the test split.

No Arabic-Digits-specific beta sweep or beta optimization is performed by Formula-SWCT.

## Environment Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The repository is cloned, the `stage-wise-ct` branch is selected, and the required research dependencies are installed.

After dependency installation, restart the runtime before continuing.

In [1]:
!nvidia-smi

%cd /content

import os

if not os.path.exists("/content/curvature-tuning-research"):
    !git clone -b stage-wise-ct https://github.com/Syed1611/curvature-tuning-research.git
else:
    print("Repository already exists; skipping clone.")

%cd /content/curvature-tuning-research
!git checkout stage-wise-ct
!git pull

%cd /content/curvature-tuning-research/src/curvature-tuning
!git branch --show-current

Tue Sep 29 23:39:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 6.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 8.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 43.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 125.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 113.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 65.6 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

### Runtime Restart Required

Restart the Colab runtime after dependency installation.

After restarting, do not rerun the installation cell. Continue from the next cell.

In [1]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning


In [2]:
import torch
import datasets
import numpy
import pandas
import sklearn
import tqdm
import loguru

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("datasets:", datasets.__version__)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("sklearn:", sklearn.__version__)
print("tqdm:", tqdm.__version__)
print("loguru:", loguru.__version__)

Torch: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
datasets: 3.4.1
NumPy: 1.26.3
Pandas: 2.2.3
sklearn: 1.5.2
tqdm: 4.66.5
loguru: 0.7.2


## Arabic Digits Dataset Setup

The Arabic Handwritten Digits train, validation, and test loaders are constructed using the project's standard transfer-learning pipeline.

This cell verifies the dataset sizes and tensor structure before running the evaluation methods.

In [3]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_arabic-digits",
    train_batch_size=32,
    test_batch_size=64,
    seed=42
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Label min:", labels.min().item())
print("Label max:", labels.max().item())

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

arabic-digits.py: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Processing train set: 100%|██████████| 60001/60001 [00:16<00:00, 3674.49it/s]


Generating test split: 0 examples [00:00, ? examples/s]

Processing test set: 100%|██████████| 10001/10001 [00:02<00:00, 4263.67it/s]


Train samples: 48000
Validation samples: 12000
Test samples: 10000
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Label min: 0
Label max: 9


# Baseline and Original S-CT Reference

The original Baseline and Single-Parameter Curvature Tuning methods are evaluated first.

The Baseline uses the frozen ImageNet-pretrained ResNet-18 with the original ReLU activations.

S-CT performs its original validation-based search for one global beta parameter shared across the network.

The surviving seed-42 experiment produced:

- Baseline test accuracy: **97.82%**
- S-CT test accuracy: **98.83%**
- Selected global beta: **0.74**
- Best S-CT validation accuracy: **99.42%**

These results provide the reference for the Formula-SWCT evaluation.

In [ ]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-digits \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 64

2026-09-27 18:34:55.051 | INFO     | __main__:main:54 - Log file: ./logs/generalization_ct_imagenet_to_arabic-digits_resnet18_seed42.log
2026-09-27 18:34:55.054 | INFO     | __main__:main:58 - Running on cuda
2026-09-27 18:36:03.014 | INFO     | __main__:main:85 - Testing baseline...
2026-09-27 18:36:03.027 | INFO     | __main__:main:88 - Number of trainable parameters: 5130
2026-09-27 18:36:03.028 | INFO     | __main__:main:89 - Starting transfer learning...
2026-09-27 18:37:47.683 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 2.491875, Accuracy: 9.38%
2026-09-27 18:37:48.079 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 2.352832, Accuracy: 9.78%
2026-09-27 18:37:48.459 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 2.214385, Accuracy: 22.82%
2026-09-27 18:37:48.727 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.050452, Accuracy: 38.55%
2026-09-27 18:37:49.016 | INFO     | train:train_epoch:47 - Epoch 1, Step 400, Loss: 1.86994

# Fit the Frozen Formula-SWCT Predictor

Because this notebook is intended to run independently in a fresh Colab runtime, the Formula-SWCT predictor is reconstructed before the Arabic Digits evaluation.

The formula is fitted using only the three development/calibration datasets:

| Dataset | Stage 1 | Stage 2 | Stage 3 | Stage 4 |
|---|---:|---:|---:|---:|
| Beans | 0.85 | 0.85 | 0.85 | 0.80 |
| DTD | 0.95 | 0.95 | 0.99 | 0.99 |
| Flowers102 | 0.99 | 0.95 | 0.99 | 0.90 |

For each calibration dataset and seed, five stage descriptors are measured from the frozen ImageNet-pretrained ResNet-18:

1. log activation variance,
2. activation sparsity,
3. log mean absolute activation,
4. log class-separation statistic,
5. normalized stage depth.

A ridge-regression predictor with lambda = 0.10 is fitted once from these observations.

Arabic Digits is not used to modify or retune the formula.

In [4]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode fit \
    --model resnet18 \
    --pretrained_ds imagenet \
    --train_bs 32 \
    --test_bs 64

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: FIT DIRECT BETA FORMULA
No beta sweep and no downstream training are performed here.

data/train-00000-of-00001.parquet: 100% 144M/144M [00:01<00:00, 86.2MB/s] 
data/validation-00000-of-00001.parquet: 100% 18.5M/18.5M [00:01<00:00, 18.4MB/s]
data/test-00000-of-00001.parquet: 100% 17.7M/17.7M [00:00<00:00, 21.9MB/s]
Generating train split: 1034 examples [00:00, 2425.50 examples/s]
Generating validation split: 133 examples [00:00, 1658.68 examples/s]
Generating test split: 128 examples [00:00, 2166.29 examples/s]
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 59.6MB/s]



100% 625M/625M [00:03<00:00, 200MB/s]  



100% 345M/345M [00:01<00:00, 243MB/s]  
100% 502/502 [00:00<00:00, 2.41MB/s]
100% 15.0k/15.0k [00:00<00:00, 77.2MB/s]



FITTED FORMULA
beta_hat = clip(b0 + sum(w_j * z_j), 0.7, 0.99)
intercept

# Final Formula-SWCT Evaluation

The fitted Formula-SWCT predictor is now applied to the Arabic Handwritten Digits dataset.

Stage statistics are collected only from the target training split.

The formula then predicts four beta values before downstream classifier training. These beta values are frozen for the remainder of the experiment.

The fixed-beta model is used to extract features once, after which one downstream linear classifier is trained.

No Arabic-Digits-specific beta search or beta optimization is performed.

In [5]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-digits \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 20

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: arabic-digits
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.990000
Stage 2: 0.990000
Stage 3: 0.966184
Stage 4: 0.895451
Predicted beta vector: [0.99, 0.99, 0.966184, 0.895451]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.99, 0.99, 0.966183, 0.895451]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=0.196716 | val_acc=96.16%
Epoch 02 | val_loss=0.104448 | val_acc=97.50%
Epoch 03 | val_loss=0.079230 | val_acc=97.90%
Epoch 04 | val_loss=0.068160 | val_acc=98.08%
Epoch 05 | val_loss=0.059262 | val_acc=98.33%
Epoch 06 | val_loss=0.058015 | val_acc=98.38%
Epoch 07 | val_loss=0.050127 | val_acc=98.58%
Epoch 08 | val_loss=0.055532 | val_acc=98.30%
Epoch 09 | val_loss=0.046049 | val_acc=98.74%
Epoch 10 | val_loss=0.041946 | val_acc=98.78%
Epoch 11 | val_los

In [6]:
%cd /content/curvature-tuning-research/src/curvature-tuning

!python formula_swct.py \
    --mode run \
    --model resnet18 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-digits \
    --seed 42 \
    --train_bs 32 \
    --test_bs 64 \
    --epochs 30

/content/curvature-tuning-research/src/curvature-tuning
FORMULA-SWCT: PREDICT FIRST, TRAIN ONCE
Dataset: arabic-digits
Seed: 42
No beta sweep. No beta optimization. No 0.78 anchor.

Predicted betas BEFORE training:
Stage 1: 0.990000
Stage 2: 0.990000
Stage 3: 0.966184
Stage 4: 0.895451
Predicted beta vector: [0.99, 0.99, 0.966184, 0.895451]
Stage-Wise CT ReLU counts: [3, 2, 2, 2]
Loaded fixed betas: [0.99, 0.99, 0.966183, 0.895451]

Extracting fixed Formula-SWCT features once...
Training ONE downstream classifier...
Epoch 01 | val_loss=0.196716 | val_acc=96.16%
Epoch 02 | val_loss=0.104448 | val_acc=97.50%
Epoch 03 | val_loss=0.079230 | val_acc=97.90%
Epoch 04 | val_loss=0.068160 | val_acc=98.08%
Epoch 05 | val_loss=0.059262 | val_acc=98.33%
Epoch 06 | val_loss=0.058015 | val_acc=98.38%
Epoch 07 | val_loss=0.050127 | val_acc=98.58%
Epoch 08 | val_loss=0.055532 | val_acc=98.30%
Epoch 09 | val_loss=0.046049 | val_acc=98.74%
Epoch 10 | val_loss=0.041946 | val_acc=98.78%
Epoch 11 | val_los

## Previously Completed Formula-SWCT Run

A complete Formula-SWCT seed-42 experiment on Arabic Handwritten Digits was previously executed.

The recorded prediction was:

**[0.990000, 0.990000, 0.966184, 0.895451]**

The corresponding evaluation produced:

- Best validation accuracy: **98.84%**
- Best validation epoch: **11**
- Test loss: **0.065744**
- Test accuracy: **97.98%**

The first two stage predictions reach the Formula-SWCT upper constraint of beta = 0.99.

This indicates that, for this dataset, the fitted formula requests very low curvature modification in the earlier network stages relative to the allowed prediction range.

The experiment will be reproduced using the execution cell above so that its generated result file and complete output are restored in this notebook.

# Arabic Digits Results Summary

For seed 42:

| Method | Beta Selection | Test Accuracy |
|---|---|---:|
| Baseline | Original ReLU | **97.82%** |
| S-CT | Validation search over one global beta | **98.83%** |
| Formula-SWCT | Direct four-stage prediction | **97.98%** |

S-CT selected:

**beta = 0.74**

with a best validation accuracy of:

**99.42%**

Formula-SWCT predicted:

**[0.990000, 0.990000, 0.966184, 0.895451]**

with:

- Best validation accuracy: **98.84%**
- Best validation epoch: **11**
- Test accuracy: **97.98%**

Relative to the Baseline, Formula-SWCT improves test accuracy by approximately:

**+0.16 percentage points**

On this dataset, the searched S-CT result remains higher than Formula-SWCT by approximately:

**0.85 percentage points**

The purpose of Formula-SWCT is different from S-CT: S-CT performs target-specific validation search over beta, whereas Formula-SWCT predicts all four beta values directly without target-specific beta optimization.